### GPT4Rec

data from amazon beauty

users: 22254, items: 11778, interactions: 190726, ave-length: 7.439, meta info: category

In [1]:
import json
import pandas as pd

In [2]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator("Hello, I'm a language model,", max_length=30, num_return_sequences=5)

/home/jylee/miniconda3/envs/torchrec/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Xformers is not installed correctly. If you want to use memorry_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': "Hello, I'm a language model, but what I'm really doing is making a human-readable document. There are other languages, but those are"},
 {'generated_text': "Hello, I'm a language model, not a syntax model. That's why I like it. I've done a lot of programming projects.\n"},
 {'generated_text': "Hello, I'm a language model, and I'll do it in no time!\n\nOne of the things we learned from talking to my friend"},
 {'generated_text': "Hello, I'm a language model, not a command line tool.\n\nIf my code is simple enough:\n\nif (use (string"},
 {'generated_text': "Hello, I'm a language model, I've been using Language in all my work. Just a small example, let's see a simplified example."}]

### Prompt

In [3]:
prompt_template = '''\
Previously, the customer has bought: 
%s 
In the future, the customer wants to buy\
'''
print(prompt_template % ', '.join(['ITEM TITLE1', 'ITEM TITLE2']))

Previously, the customer has bought: 
ITEM TITLE1, ITEM TITLE2 
In the future, the customer wants to buy


In [4]:
def input_transform(user_history):
    return prompt_template % ', '.join(user_history)

### Prepare Data

In [5]:
df_rating = pd.read_csv('../data/amazon_beauty/ratings_Beauty.csv')
df_rating.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2023070 entries, 0 to 2023069
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   UserId     object 
 1   ProductId  object 
 2   Rating     float64
 3   Timestamp  int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 61.7+ MB


user - item1, item2, item3 ...  이런 식으로 변경

In [6]:
len(df_rating.UserId.unique()), len(df_rating.ProductId.unique()), df_rating.shape[0]

(1210271, 249274, 2023070)

In [7]:
print(df_rating.shape[0])
df_rating.drop_duplicates(subset=['UserId', 'ProductId'], inplace=True)
print(df_rating.shape[0])

2023070
2023070


In [8]:
valid_users = df_rating['UserId'].value_counts()[df_rating['UserId'].value_counts() >= 5].index
df_rating2 = df_rating[df_rating['UserId'].isin(valid_users)]
valid_items = df_rating2['ProductId'].value_counts()[df_rating2['ProductId'].value_counts() >= 5].index
df_rating2 = df_rating2[df_rating2['ProductId'].isin(valid_items)]
print(len(valid_users), len(valid_items), df_rating2.shape)

num_users, num_items = 0,0
for i in range(20):
    valid_users = df_rating2['UserId'].value_counts()[df_rating2['UserId'].value_counts() >= 5].index
    df_rating2 = df_rating2[df_rating2['UserId'].isin(valid_users)]
    valid_items = df_rating2['ProductId'].value_counts()[df_rating2['ProductId'].value_counts() >= 5].index
    df_rating2 = df_rating2[df_rating2['ProductId'].isin(valid_items)]

    if num_users == len(valid_users) & num_items == len(valid_items):
        break
    else:
        num_users, num_items = len(valid_users), len(valid_items)

    print(i, len(valid_users), len(valid_items), df_rating2.shape)

52374 19369 (313823, 4)
27501 13727 (224229, 4)
23746 12562 (205760, 4)
22787 12247 (200771, 4)
22505 12153 (199277, 4)
22408 12116 (198741, 4)
22374 12103 (198554, 4)
22364 12101 (198506, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)
22363 12101 (198502, 4)


In [9]:
df_rating2 = df_rating2.sort_values(by=['UserId', 'Timestamp'])
df_rating2.head()

,UserId,ProductId,Rating,Timestamp
1581545,A00414041RD0BXM6WK0GX,B007IY97U0,3.0,1405296000
1643683,A00414041RD0BXM6WK0GX,B00870XLDS,2.0,1405296000
1681280,A00414041RD0BXM6WK0GX,B008MIRO88,1.0,1405296000
1853091,A00414041RD0BXM6WK0GX,B00BQYYMN0,3.0,1405296000
1975026,A00414041RD0BXM6WK0GX,B00GRTQBTM,5.0,1405296000


In [10]:
# Code provided via http://jmcauley.ucsd.edu/data/amazon/

def parse_gz(path):
    # g = gzip.open(path, 'rb')
    f = open(path, 'r')
    for l in f:
        yield eval(l)

def convert_to_DF(path):
    i = 0
    df = {}
    for d in parse_gz(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

In [11]:
beauty = convert_to_DF('../data/amazon_beauty/meta_Beauty.json')

In [12]:
beauty.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 259204 entries, 0 to 259203
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   asin         259204 non-null  object 
 1   description  234497 non-null  object 
 2   title        258760 non-null  object 
 3   imUrl        259116 non-null  object 
 4   salesRank    254016 non-null  object 
 5   categories   259204 non-null  object 
 6   price        189930 non-null  float64
 7   related      207854 non-null  object 
 8   brand        128166 non-null  object 
dtypes: float64(1), object(8)
memory usage: 19.8+ MB


In [13]:
print("before drop duplicates: ", beauty.shape[0])
beauty.drop_duplicates(subset='asin', inplace=True)
print("after drop duplicates: ", beauty.shape[0])

before drop duplicates:  259204
after drop duplicates:  259204


In [14]:
print("before drop duplicates: ", beauty.shape[0])
beauty.drop_duplicates(subset='title', inplace=True)
print("after drop duplicates: ", beauty.shape[0])

before drop duplicates:  259204
after drop duplicates:  254925


In [18]:
import matplotlib.pyplot as plt
# more than 400 characters
print("before drop: ", beauty.shape[0])
beauty.dropna(subset=['title'], axis=0, how='any', inplace=True)
print("dropna: ", beauty.shape[0])
beauty['title'] = beauty['title'].apply(lambda x: x[:400])
print("truncated <= 400: ", beauty.shape[0])
beauty.dropna(subset='title', inplace=True)


before drop:  254924
dropna:  254924
truncated <= 400:  254924


In [19]:
beauty.title.apply(len).mean()

62.50886538733113

# 병합

In [20]:
df_rating = pd.merge(
    df_rating2, beauty[['asin', 'title']], left_on='ProductId', 
    right_on='asin', how='inner').drop(columns='asin')
df_rating.shape

(197032, 5)

In [21]:
movie_map = {p:i for i,p in enumerate(df_rating['UserId'])}
user_map = {u:i for i,u in enumerate(df_rating['ProductId'])}

len(user_map.keys()), len(movie_map.keys()), df_rating.shape[0]

(11984, 22363, 197032)

In [22]:
df_rating.groupby('UserId')['ProductId'].apply(list).values

array([list(['B007IY97U0', 'B00870XLDS', 'B008MIRO88', 'B00BQYYMN0', 'B00GRTQBTM', 'B00HFP4JZU']),
       list(['B0019LVFI0', 'B0020HEBX8', 'B006R5GXCG', 'B001L2BEWE', 'B00B18C2RE', 'B000052YQU']),
       list(['B001RMP7M6', 'B003TMO3EU', 'B00028M3N2', 'B0035RF85C', 'B005Y5VL4U', 'B006GK5NNW', 'B007EZ0CC0', 'B0091JL3IO']),
       ...,
       list(['B000WYZ9Q4', 'B0032CDFCS', 'B002LFLPEC', 'B003BMJGL8', 'B00GLS5DKM', 'B007O7AZBG', 'B004TSFE6Y', 'B004G7XZTG', 'B006ZBP8NM', 'B00C7DYBX0', 'B00AVUE1S6', 'B002DP1A18', 'B00C64GX9U', 'B003BOM2JY', 'B001L433TO', 'B009ERIUNY']),
       list(['B008PGD4UO', 'B0013JSK7M', 'B00BAK7JTE', 'B003ZZOUYY', 'B0042PE8LQ']),
       list(['B000QE5GU4', 'B00027DMSI', 'B004CQ710U', 'B005XP4YNQ', 'B006ZUEMSA', 'B005RFI1YK'])],
      dtype=object)

In [23]:
print(f"num of users: {df_rating['UserId'].unique().shape[0]}")
print(f"num of products: {df_rating['ProductId'].unique().shape[0]}")

num of users: 22363
num of products: 11984


# 여기부터 해야됨

### split train:val:test into 0.8:0.1:0.1

In [24]:
from sklearn.model_selection import train_test_split

In [25]:
users = df_rating['UserId'].unique()
users

array(['A00414041RD0BXM6WK0GX', 'A10Q5LBB5HQ0BJ', 'A1HW72TSGGCGOZ', ...,
       'A32TCWAILP09A2', 'A3851PGBIQN5MY', 'A3TCUVQJOKFWZ7'], dtype=object)

In [26]:
import numpy as np

users.shape, users[df_rating['UserId'].value_counts() >= 2].shape

((22363,), (22363,))

array(['A00414041RD0BXM6WK0GX', 'A10Q5LBB5HQ0BJ', 'A1HW72TSGGCGOZ', ...,
       'A32TCWAILP09A2', 'A3851PGBIQN5MY', 'A3TCUVQJOKFWZ7'], dtype=object)